In [0]:
# ==========================================
# FACT NOTEBOOK (BATCH VERSION)
# ==========================================

from pyspark.sql.functions import *

# ------------------------------------------
# TABLES
# ------------------------------------------

silver_table = "telecom_rajeswari_workspace.banking_silver.transactions_silver_tbl"

dim_account_table = "telecom_rajeswari_workspace.banking_gold.dim_account_gold_scd1"

dim_customer_table = "telecom_rajeswari_workspace.banking_gold.dim_customer_gold_scd1"

gold_fact_table = "telecom_rajeswari_workspace.banking_gold.fact_wide_transactions"

gold_aggr_table = "telecom_rajeswari_workspace.banking_gold.fact_account_transactions"

# ------------------------------------------
# READ TABLES (BATCH)
# ------------------------------------------

silver_df = spark.table(silver_table)

dim_account = (
    spark.table(dim_account_table)
    .select(
        "ACCOUNT_ID",
        "CUST_ID",
        "TYPE",
        "BALANCE",
        "STATUS"
    )
)

dim_customer = (
    spark.table(dim_customer_table)
    .select(
        "CUST_ID",
        "NAME_MASKED",
        "Risk_Score",
        "KYC_Status"
    )
)

# ------------------------------------------
# FACT TABLE
# ------------------------------------------

fact_df = (

    silver_df.alias("f")

    .join(
        dim_account.alias("a"),
        col("f.ACCOUNT_ID")==col("a.ACCOUNT_ID"),
        "left"
    )

    .join(
        dim_customer.alias("c"),
        col("a.CUST_ID")==col("c.CUST_ID"),
        "left"
    )

    .select(

        col("f.txn_id"),
        col("f.ACCOUNT_ID"),
        col("f.card_id"),
        col("f.amount"),
        col("f.merchant_cat"),
        col("f.transaction_timestamp"),
        col("f.transaction_date"),
        col("f.transaction_hour"),
        col("f.transaction_status"),
        col("f.amount_bucket"),
        col("f.silver_load_time"),

        col("a.CUST_ID"),
        col("c.NAME_MASKED"),
        col("c.Risk_Score"),
        col("c.KYC_Status"),

        col("a.TYPE"),
        col("a.BALANCE"),
        col("a.STATUS"),

        current_timestamp().alias("gold_load_time")

    )

)

# ------------------------------------------
# WRITE FACT TABLE
# ------------------------------------------

(
    fact_df.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema","true")
    .saveAsTable(gold_fact_table)
)

print("Fact Table Loaded")

# ------------------------------------------
# AGGREGATION
# ------------------------------------------

aggr_df = (

    silver_df

    .groupBy(
        "transaction_date",
        "ACCOUNT_ID"
    )

    .agg(

        count("*").alias("total_transactions"),

        sum("amount").alias("total_amount"),

        avg("amount").alias("average_amount"),

        sum(
            when(
                col("transaction_status")=="SUCCESS",
                1
            ).otherwise(0)
        ).alias("successful_transactions")

    )

)

# ------------------------------------------
# WRITE AGGREGATION
# ------------------------------------------

(
    aggr_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(gold_aggr_table)
)

print("Aggregation Table Loaded")

print("Batch Fact Notebook Completed Successfully")

In [0]:
%sql
SELECT COUNT(*) 
FROM telecom_rajeswari_workspace.banking_gold.fact_wide_transactions;

In [0]:
%sql
SELECT COUNT(*) 
FROM telecom_rajeswari_workspace.banking_gold.fact_account_transactions;